<a href="https://colab.research.google.com/github/Shriniwas18K/Resources/blob/main/LLMs/Spacy_for_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
text = "Tech is fascinating"
text

'Tech is fascinating'

#### Mainly Docs, Tokens, Spans are three main objects given by Pipeline

In [ ]:
doc = nlp(text)

## Grammer findings

### tokenization and POS tagging of each token
- coarse grained tags: pos_ : based on Universal Dependencies
- fine grained tage: tag_ : based on text we gave as input

In [ ]:
for token in doc:
  print(token, token.pos_, token.tag_)

Tech NOUN NN
is AUX VBZ
fascinating ADJ JJ


### morphology
- Number: Sing(ular) , Plur(al)
- PronType: Art(icle)
- VerbForm: Fin(al), Part(icple)
- Tense: Pres(ent), Past, Fut(ure)

In [ ]:
for token in doc:
  print(token,token.morph.to_dict())

Tech {'Number': 'Sing'}
is {'Mood': 'Ind', 'Number': 'Sing', 'Person': '3', 'Tense': 'Pres', 'VerbForm': 'Fin'}
fascinating {'Degree': 'Pos'}


### Syntactic deps b/w tokens

In [ ]:
token = doc[0]

print(doc)
print(token) # token as child
print(token.head) # parent token
print(token.i) # index of token as child
print(token.head.i) # parent token index
print(token.dep_) # relation between parent and child token

Tech is fascinating
Tech
is
0
1
nsubj


#### The syntactic relationship can be visualised using Spacy

In [ ]:
from spacy import displacy

displacy.render(doc,style='dep',options={"compact":True})

### doc object can hold longer texts and we can use sentences in it

In [ ]:
for sentence in doc.sents:
  print(sentence)

Tech is fascinating


### convert token to its base form:
lemmatisation: uses ML underhood whereas stemming does not use ML

In [ ]:
for token in doc:
  print(token,token.lemma_)

Tech tech
is be
fascinating fascinating


### named entity recognition over entire doc

In [ ]:
for entity in doc.ents:
  print(entity.text,entity.label_)

Tech ORG


In [ ]:
displacy.render(doc,style='ent')

### spacy has inbuilt docs using explain function, to explain any tag

In [ ]:
spacy.explain("ORG")

'Companies, agencies, institutions, etc.'

### Customising Spacy Pipelines

view the pipeline

In [ ]:
nlp.pipeline

[('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7df060f6ccb0>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7df060f6d8b0>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7df060fb7140>),
 ('attribute_ruler',
  <spacy.pipeline.attributeruler.AttributeRuler at 0x7df060dac750>),
 ('lemmatizer',
  <spacy.lang.en.lemmatizer.EnglishLemmatizer at 0x7df060daf1d0>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x7df060fb70d0>)]

all pipeline components have compute involved, hence only use the ones needed

In [ ]:
nlp2 = spacy.load("en_core_web_sm",exclude = ["ner","parser"])
nlp2.pipeline

[('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7df05fcce810>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7df05fcce3f0>),
 ('attribute_ruler',
  <spacy.pipeline.attributeruler.AttributeRuler at 0x7df05e7b9790>),
 ('lemmatizer',
  <spacy.lang.en.lemmatizer.EnglishLemmatizer at 0x7df05e783c50>)]

tokenizer is inside seprate attribute because its always needed

In [ ]:
nlp2.tokenizer

pipe analysis to check wheter any pipe component has problem

In [ ]:
pipe_analysis = nlp2.analyze_pipes(pretty=True)
pipe_analysis


============================= Pipeline Overview =============================

#   Component         Assigns       Requires   Scores        Retokenizes
-   ---------------   -----------   --------   -----------   -----------
0   tok2vec           doc.tensor                             False      
                                                                        
1   tagger            token.tag                tag_acc       False      
                                               pos_acc                  
                                               tag_micro_p              
                                               tag_micro_r              
                                               tag_micro_f              
                                                                        
2   attribute_ruler                                          False      
                                                                        
3   lemmatizer        token.lemma           

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc',
    'pos_acc',
    'tag_micro_p',
    'tag_micro_r',
    'tag_micro_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False}},
 'problems': {'tok2vec': [],
  'tagger': [],
  'attribute_ruler': [],
  'lemmatizer': []},
 'attrs': {'token.tag': {'assigns': ['tagger'], 'requires': []},
  'doc.tensor': {'assigns': ['tok2vec'], 'requires': []},
  'token.lemma': {'assigns': ['lemmatizer'], 'requires': []}}}

assert for runtime unit testing and debugging

In [ ]:
for component_name,problem_list in pipe_analysis['problems'].items():
  assert len(problem_list)==0, f"Check {component_name}"

### On demand or batch processing of sentences using .pipe() which returns generator

In [ ]:
sents = ["Hello world","Happy new year","Binary Search Trees are best for storing items"]
for doc in nlp2.pipe(sents):
  print(doc)

Hello world
Happy new year
Binary Search Trees are best for storing items


### Custom attributes to docs: those are stored under _ attribute of document, it is preferred to use this rather than wrapping the doc object because docs can be stored into DocBin when store_user_data=True

In [ ]:
from spacy.tokens import Doc
Doc.set_extension('age',default=100)

In [ ]:
docs = []
texts = [
    "Hello world",
    "I am is you",
    "Happy Birthday"
]
for idx,sentence in enumerate(texts):
    doc = nlp(sentence)
    doc._.age = idx+100
    docs.append(doc)

### DocBin to store docs to disk

In [ ]:
from spacy.tokens import DocBin
docbin=DocBin(docs=docs,store_user_data=True)

In [ ]:
docbin.add(nlp("Python Spacy"))

In [ ]:
len(docbin)

4

In [ ]:
docbin.to_disk("/temp.spacy")

In [ ]:
docbin_loaded = DocBin().from_disk("/temp.spacy")
loaded_docs = docbin_loaded.get_docs(nlp.vocab)
for doc in loaded_docs:
  print(doc)

Hello world
I am is you
Happy Birthday
Python Spacy


### Merging noun words into single noun
- sometimes some nouns have multiple words hence instead of processing them in multiple of tokens we should use single token
- each noun_chunk is span, having start end as indices

In [ ]:
text = "Virat Kohli is awarded by the government of India by Padma Bhushan."
for noun_chunk in nlp(text).noun_chunks:
  print(noun_chunk,noun_chunk.start,noun_chunk.end)

Virat Kohli 0 2
the government 5 7
India 8 9
Padma Bhushan 10 12


In [ ]:
displacy.render(nlp(text),style='dep',options={"compact":True})

In [ ]:
nlp.add_pipe("merge_noun_chunks")

<function spacy.pipeline.functions.merge_noun_chunks(doc: spacy.tokens.doc.Doc) -> spacy.tokens.doc.Doc>

#### each noun chunk becomes single token

In [ ]:
displacy.render(nlp(text),style='dep',options={"compact":True})

### Similarly we can merge Named Entities

In [ ]:
nlp.remove_pipe("merge_noun_chunks")

('merge_noun_chunks',
 <function spacy.pipeline.functions.merge_noun_chunks(doc: spacy.tokens.doc.Doc) -> spacy.tokens.doc.Doc>)

In [ ]:
displacy.render(nlp(text),style="ent")

In [ ]:
nlp.add_pipe("merge_entities")
displacy.render(nlp(text),style='ent')

### Evaluating annotations of datasets b/w two annotators
ground truth(aka gold standard) given by human and predictions by language model, by using cohen kappa score( nearer to 1 is better means both predicted and actual are in complete agreement)

Example: done for morphological analysis

In [ ]:
language_model_predictions = ['']
for token in nlp(text):
    language_model_predictions.append(token.pos_)
import random
ground_truth = language_model_predictions.copy()
random.shuffle(ground_truth) # for simplicity

from sklearn.metrics import cohen_kappa_score
print(cohen_kappa_score(y1=ground_truth,y2=language_model_predictions))

-0.1304347826086958


### Confusion matrix
precision recall f1 support, everything comes in classification report

In [ ]:









from sklearn.metrics import confusion_matrix
print(confusion_matrix(ground_truth,language_model_predictions))

[[0 0 0 1 0 0 0 0]
 [0 0 0 0 0 3 0 0]
 [0 0 0 0 0 0 0 1]
 [0 1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 2 1 0 0 0 1 0]
 [0 0 0 0 0 1 0 0]
 [1 0 0 0 0 0 0 0]]


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ground_truth,language_model_predictions))

              precision    recall  f1-score   support

                   0.00      0.00      0.00         1
         ADP       0.00      0.00      0.00         3
         AUX       0.00      0.00      0.00         1
         DET       0.00      0.00      0.00         1
        NOUN       1.00      1.00      1.00         1
       PROPN       0.00      0.00      0.00         4
       PUNCT       0.00      0.00      0.00         1
        VERB       0.00      0.00      0.00         1

    accuracy                           0.08        13
   macro avg       0.12      0.12      0.12        13
weighted avg       0.08      0.08      0.08        13



### Universal dependencies
- it is framework for analyzing grammer across various languages
- each token is word and is in one of three types
- nominal: nouns, representing things
- clauses: verbs representing actions and events
- modifiers: adjectives adverbs collaborating with above two

Spacy supports Universal dependencies as the tags given by it belong to UD framework

### Some other things
- we can find pronoun+verb etc grammatical combinations from text using spacy
- we can use Stanza library for other languages
- CoNLL-U is extension of UD framework and used to represent plain text with grammer, but not builtin into Spacy, the doc,span,token, objects hence augment _ attribute of them

## word embeddings
distributional hypothesis: "You shall know a word by the company it keeps".

#### Syntagmatic perspective:
- number of dimensions equals number of distinct words, sparsity issues
- sentences having more words in common are similar
- compare one hot encoded vectors of sentences with cosine similarity or Jaccard similarity
- context is not taken into account

### Paradigmitic perspective:
- co-occurence matrix of words considers number of times the word in column is neighbor(i.e. within +-2 indexes of word position in text) of the word in row
- cosine similarity to measure similarity between words based on the context
- sparsity issues

Based on above approaches we can take vector of word and pass it to neural network and predict another word by supervised training by giving labels as most probable next occuring word or similar word

Spacy provides builtin word embedding models trained on large corpora of words, which we can get by using large models, just check vector attribute of each token,span. Doc embeddings are average of embeddings of all tokens in the doc.

In [ ]:
! python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp_emb = spacy.load("en_core_web_lg")

if small models like en_core_web_sm are used then the embeddings are dependent on taggers,ners,tokenizers only, their are no syntactic or paradigmatic embeddings concept

In [ ]:
for token in nlp_emb("Hello world"):
  print(token,"\t",token.vector[:5])

Hello 	 [ 0.25233  0.10176 -0.67485  0.21117  0.43492]
world 	 [-0.0066796  0.22238    0.27709   -0.1676     0.39934  ]


cosine similarity check

In [ ]:
hello,world = nlp_emb("Hello world")
hello.similarity(world)

0.2545972764492035

model vocab words are known as lexical types because they are static, i.e their context is only in training data, whenever they are given for prediction to model

the words coming in input are called as tokens as their context depends on sentence too. "bank of river" "bank of America".

the contextual word or sentence embeddings can be captured by transformers architectures like BERT GPT-3, which can be used in spacy

In [ ]:
! python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 2.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
nlp_trf = spacy.load("en_core_web_trf")

In [ ]:
doc = nlp_trf("Hello world")
doc._.trf_data

DocTransformerOutput(all_outputs=[Ragged(data=array([[ 0.6816944 , -0.45504525,  0.79676   , ..., -0.4970303 ,
        -0.82392603, -0.6464289 ],
       [-0.01352241, -0.9686055 , -0.5448992 , ..., -1.2530415 ,
        -0.62953436, -0.23707493]], dtype=float32), lengths=array([1, 1], dtype=int32), data_shape=(-1, 768), starts_ends=None)], last_layer_only=True)

In [ ]:
doc._.trf_data

DocTransformerOutput(all_outputs=[Ragged(data=array([[ 0.6816944 , -0.45504525,  0.79676   , ..., -0.4970303 ,
        -0.82392603, -0.6464289 ],
       [-0.01352241, -0.9686055 , -0.5448992 , ..., -1.2530415 ,
        -0.62953436, -0.23707493]], dtype=float32), lengths=array([1, 1], dtype=int32), data_shape=(-1, 768), starts_ends=None)], last_layer_only=True)